# Pipeline Visualization - Step 1: Input Dataset

This notebook creates the first visual asset for the report pipeline: a GTSRB input sample with its ROI annotation. The output figure is intended to be used in the `Input Data` block of the methodology diagram.

## What the first pipeline image should show

For the input dataset block, use one original GTSRB training image with the provided ROI bounding box drawn on top. This is better than showing only a generic database icon because the actual method receives both the image and its annotation fields: filename, image size, ROI coordinates, and class label.

In [ ]:
from pathlib import Path
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

TRAIN_IMAGES_DIR = Path(r"C:\Users\carwynetic\Downloads\drive-download-20260713T142225Z-2-001\GTSRB_Final_Training_Images\GTSRB\Final_Training\Images")
OUTPUT_DIR = Path(r"C:\Users\carwynetic\OneDrive\Documents\CPV\pipeline_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Training images directory exists:", TRAIN_IMAGES_DIR.exists())
print("Output directory:", OUTPUT_DIR)

## Select one clear example

Class `14` is used here because it is the `Stop` sign class in GTSRB and usually gives a clear visual example. If you prefer another class, change `CLASS_ID`.

In [ ]:
CLASS_ID = 14
class_dir = TRAIN_IMAGES_DIR / f"{CLASS_ID:05d}"
csv_path = class_dir / f"GT-{CLASS_ID:05d}.csv"

df = pd.read_csv(csv_path, sep=";")
sample = df.iloc[len(df) // 2]

image_path = class_dir / sample["Filename"]
roi = (
    int(sample["Roi.X1"]),
    int(sample["Roi.Y1"]),
    int(sample["Roi.X2"]),
    int(sample["Roi.Y2"]),
)

print("CSV:", csv_path)
print("Image:", image_path)
print("ClassId:", int(sample["ClassId"]))
print("ROI:", roi)
display(sample.to_frame().T)

## Draw the input dataset figure

The red rectangle is the ROI from the GTSRB annotation CSV. For the final draw.io diagram, place this image under or inside the `Input Data` block.

In [ ]:
img_bgr = cv2.imread(str(image_path))
if img_bgr is None:
    raise FileNotFoundError(image_path)

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
x1, y1, x2, y2 = roi

fig, ax = plt.subplots(figsize=(4.2, 3.2), dpi=200)
ax.imshow(img_rgb)
ax.add_patch(
    Rectangle(
        (x1, y1),
        x2 - x1 + 1,
        y2 - y1 + 1,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
    )
)
ax.set_title("Input image with ROI annotation", fontsize=10)
ax.text(
    0.5,
    -0.12,
    f"Filename: {sample['Filename']} | ClassId: {int(sample['ClassId'])} | ROI: ({x1}, {y1}, {x2}, {y2})",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=7,
)
ax.axis("off")
fig.tight_layout()

out_path = OUTPUT_DIR / "step1_input_dataset_roi.png"
fig.savefig(out_path, bbox_inches="tight", pad_inches=0.08)
plt.show()

print("Saved:", out_path)